# DistilBERT Single-Model (Label1-first thresholding) — End-to-End
Generated: 2025-10-18T00:26:50

In [ ]:

# ===== 0) Setup (installs) =====
%pip -q install "transformers>=4.44" "datasets>=2.20" "evaluate>=0.4.2" "accelerate>=0.33" scikit-learn matplotlib --upgrade


In [ ]:

# ===== 1) Imports & config =====
import os, random, math, json
from pathlib import Path
from typing import List, Dict, Any
import numpy as np
import torch
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments,
                          EarlyStoppingCallback, pipeline)
import evaluate
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score, precision_score
import matplotlib.pyplot as plt
from collections import Counter

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

MODEL_CKPT = "distilbert-base-uncased"
TEXT_COL = "text"
LABEL_COL = "label"

# Provided mapping: Label4->0, Label5->1, Label2->2, Label3->3, Label1->4
id2label = {0:"Label4", 1:"Label5", 2:"Label2", 3:"Label3", 4:"Label1"}
label2id = {v:k for k,v in id2label.items()}
LABEL1_ID = 4

BATCH_SIZE = 24
GRAD_ACCUM = 1
LR = 3e-5
NUM_EPOCHS = 2
PATIENCE = 2
LABEL1_EXTRA_WEIGHT = 1.3
USE_TWO_STAGE_HEAD_WARMUP = True
HEAD_WARMUP_EPOCHS = 1

OUTPUT_DIR = "runs_distilbert_label1_single"
Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)


In [ ]:

# ===== 2) Load data =====
CSV_TRAIN = []  # fill with paths if available
CSV_VAL   = []
CSV_TEST  = []

if len(CSV_TRAIN) > 0:
    files = {"train": CSV_TRAIN}
    if len(CSV_VAL) > 0: files["validation"] = CSV_VAL
    if len(CSV_TEST) > 0: files["test"] = CSV_TEST
    ds = load_dataset("csv", data_files=files)
else:
    ds = load_dataset("yelp_review_full")
    ds = ds.rename_columns({"text": TEXT_COL, "label": LABEL_COL})

if "validation" not in ds:
    split = ds["train"].train_test_split(test_size=0.1, seed=SEED)
    data = DatasetDict(train=split["train"], validation=split["test"])
    data["test"] = ds["test"] if "test" in ds else data["validation"].train_test_split(test_size=0.5, seed=SEED)["test"]
else:
    data = ds

def coerce_labels(example):
    val = example[LABEL_COL]
    if isinstance(val, str):
        return {LABEL_COL: label2id.get(val, val)}
    else:
        return example

data = data.map(coerce_labels)
print(data)
print("Train label counts:", Counter(data["train"][LABEL_COL]))
print("Val   label counts:", Counter(data["validation"][LABEL_COL]))


In [ ]:

# ===== 3) Tokenization =====
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)
def tok(batch):
    return tokenizer(batch[TEXT_COL], truncation=True)
data_tok = data.map(tok, batched=True, remove_columns=[TEXT_COL])
collator = DataCollatorWithPadding(tokenizer=tokenizer)
num_labels = len(id2label)


In [ ]:

# ===== 4) Build model =====
from transformers import AutoModelForSequenceClassification
def build_model(num_labels, freeze_encoder=False):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CKPT, num_labels=num_labels, id2label=id2label, label2id=label2id
    )
    if freeze_encoder:
        for p in model.distilbert.parameters():
            p.requires_grad = False
    return model


In [ ]:

# ===== 5) Custom Trainer (bias CE toward Label1) =====
from transformers import Trainer
import torch

custom_weights = torch.ones(num_labels, dtype=torch.float)
custom_weights[LABEL1_ID] = LABEL1_EXTRA_WEIGHT

class Label1BiasedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.logits
        loss = torch.nn.functional.cross_entropy(
            logits, labels, weight=custom_weights.to(logits.device)
        )
        return (loss, outputs) if return_outputs else loss


In [ ]:

# ===== 6) Metrics (focus on Label1) =====
metric_acc = evaluate.load("accuracy")
def compute_metrics_label1(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    out = {}
    out["accuracy"] = metric_acc.compute(predictions=preds, references=labels)["accuracy"]
    y_true_bin = (labels == LABEL1_ID).astype(int)
    y_pred_bin = (preds == LABEL1_ID).astype(int)
    out["label1_recall"] = recall_score(y_true_bin, y_pred_bin, zero_division=0)
    out["label1_precision"] = precision_score(y_true_bin, y_pred_bin, zero_division=0)
    out["label1_f1"] = f1_score(y_true_bin, y_pred_bin, zero_division=0)
    return out


In [ ]:

# ===== 7) Train =====
callbacks = [EarlyStoppingCallback(early_stopping_patience=PATIENCE)]
def make_args(run_name, epochs):
    return TrainingArguments(
        output_dir=os.path.join(OUTPUT_DIR, run_name),
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LR,
        num_train_epochs=epochs,
        evaluation_strategy="steps",
        save_strategy="steps",
        logging_steps=500,
        eval_steps=1000,
        save_steps=1000,
        metric_for_best_model="label1_f1",
        greater_is_better=True,
        load_best_model_at_end=True,
        seed=SEED,
        fp16=torch.cuda.is_available(),
        gradient_accumulation_steps=GRAD_ACCUM,
        report_to="none"
    )

if USE_TWO_STAGE_HEAD_WARMUP:
    model1 = build_model(num_labels, freeze_encoder=True)
    args1 = make_args("stage1_head_only", HEAD_WARMUP_EPOCHS)
    trainer1 = Label1BiasedTrainer(
        model=model1, args=args1,
        train_dataset=data_tok["train"],
        eval_dataset=data_tok["validation"],
        tokenizer=tokenizer, data_collator=collator,
        compute_metrics=compute_metrics_label1,
        callbacks=callbacks
    )
    trainer1.train()
    model2 = build_model(num_labels, freeze_encoder=False)
    model2.load_state_dict(model1.state_dict(), strict=False)
else:
    model2 = build_model(num_labels, freeze_encoder=False)

args2 = make_args("stage2_full", NUM_EPOCHS)
trainer = Label1BiasedTrainer(
    model=model2, args=args2,
    train_dataset=data_tok["train"],
    eval_dataset=data_tok["validation"],
    tokenizer=tokenizer, data_collator=collator,
    compute_metrics=compute_metrics_label1,
    callbacks=callbacks
)
train_out = trainer.train()
print("Best model path:", trainer.state.best_model_checkpoint)


In [ ]:

# ===== 8) Eval (val/test) =====
val_metrics = trainer.evaluate(eval_dataset=data_tok["validation"])
print("Validation:", val_metrics)
test_metrics = trainer.evaluate(eval_dataset=data_tok["test"])
print("Test:", test_metrics)


In [ ]:

# ===== 9) Tune τ for Label1-first rule on validation =====
from scipy.special import softmax

def predict_with_label1_bias(logits, tau, label1_id):
    p = softmax(logits, axis=1)
    preds = np.argmax(logits, axis=1).copy()
    mask = p[:, label1_id] >= tau
    preds[mask] = label1_id
    return preds

preds_val = trainer.predict(data_tok["validation"])
val_logits, val_labels = preds_val.predictions, preds_val.label_ids

best = (-1, None, None)
for tau in np.linspace(0.30, 0.80, 11):
    y_hat = predict_with_label1_bias(val_logits, tau, LABEL1_ID)
    y_true_bin = (val_labels == LABEL1_ID).astype(int)
    y_pred_bin = (y_hat == LABEL1_ID).astype(int)
    f1 = f1_score(y_true_bin, y_pred_bin, zero_division=0)
    rec = recall_score(y_true_bin, y_pred_bin, zero_division=0)
    score = f1
    if score > best[0]:
        best = (score, tau, (f1, rec))
best_tau = best[1]
print("Best τ on validation:", best_tau, "with (F1, Recall)=", best[2])


In [ ]:

# ===== 10) Final report + confusion matrix =====
preds_test = trainer.predict(data_tok["test"])
test_logits, test_labels = preds_test.predictions, preds_test.label_ids
y_hat_bias = predict_with_label1_bias(test_logits, best_tau, LABEL1_ID)

print("Classification report (test, Label1-biased):")
print(classification_report(test_labels, y_hat_bias, target_names=[id2label[i] for i in range(num_labels)], digits=3))

cm = confusion_matrix(test_labels, y_hat_bias, labels=list(range(num_labels)))
fig, ax = plt.subplots(figsize=(6,6))
im = ax.imshow(cm, interpolation="nearest")
ax.set_title("Confusion Matrix — Test (Label1-biased)")
plt.colorbar(im)
ax.set_xticks(range(num_labels)); ax.set_yticks(range(num_labels))
ax.set_xticklabels([id2label[i] for i in range(num_labels)], rotation=45, ha="right")
ax.set_yticklabels([id2label[i] for i in range(num_labels)])
plt.tight_layout(); plt.show()


In [ ]:

# ===== 11) Top-10 highest/lowest per-example loss on validation =====
logits = torch.from_numpy(val_logits)
labels = torch.from_numpy(val_labels).long()
per_ex_loss = torch.nn.functional.cross_entropy(logits, labels, reduction="none").numpy()

val_ds_raw = data["validation"]
idx_sorted_high = np.argsort(-per_ex_loss)[:10]
idx_sorted_low  = np.argsort(per_ex_loss)[:10]

def preview(indices, title):
    print(title)
    for i in indices:
        pred_i = int(np.argmax(val_logits[i]))
        print(f"[idx={i}] loss={per_ex_loss[i]:.4f}  true={id2label[int(val_labels[i])]}  pred={id2label[pred_i]}")
        if TEXT_COL in val_ds_raw.column_names:
            txt = val_ds_raw[i][TEXT_COL]
            print(str(txt)[:400].replace("\n"," "))
        print("-"*80)

preview(idx_sorted_high, "Top-10 Highest-Loss (Validation)")
preview(idx_sorted_low,  "Top-10 Lowest-Loss (Validation)")


In [ ]:

# ===== 12) Save model + pipeline inference examples =====
SAVE_DIR = os.path.join(OUTPUT_DIR, "best_single_model")
os.makedirs(SAVE_DIR, exist_ok=True)
trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

clf = pipeline("text-classification",
               model=SAVE_DIR, tokenizer=SAVE_DIR,
               return_all_scores=True, device=0 if torch.cuda.is_available() else -1)

samples = [
    "Outstanding service and quick resolution. Totally satisfied.",
    "It crashed again after the latest update; unusable for me.",
    "Please refund; the item arrived broken and late."
]

for s in samples:
    out = clf(s)[0]
    probs = np.array([d['score'] for d in out], dtype=float)
    pred = int(np.argmax(probs))
    if probs[LABEL1_ID] >= best_tau:
        pred = LABEL1_ID
    print("TEXT:", s)
    print("PRED:", id2label[pred], " | Label1 prob:", probs[LABEL1_ID])
    print()
